# Visualizing MJLab policies (+ velocity probes)

Loads an rl_games checkpoint trained on an MJLab task, renders a rollout to video
inline, and runs a **command-tracking probe** — commanded vs achieved base velocity.

Probes matter: on curriculum locomotion tasks, episode reward can stay high while the
policy quietly stops executing parts of the command range (reward meters average over
the curriculum distribution; a probe pins one command and measures what the robot
actually does). Use both, trust the probe.

Expectations by training scale: the notebook-scale checkpoint (2048 envs, 1000 epochs) typically achieves ~0.5-0.8 m/s at commanded 1.0; with too few envs (e.g. 512) Go1 converges to a reward-farming stander that this probe exposes at ~0.0 m/s; full-scale training (docs/MJLAB.md) reaches 0.9+.

In [1]:
CHECKPOINT = 'runs/MJLab_Go1_notebook/nn/MJLab_Go1_notebook.pth'  # <- your checkpoint
CONFIG     = '../rl_games/configs/mjlab/ppo_go1_velocity.yaml'
TASK       = 'Mjlab-Velocity-Flat-Unitree-Go1'
NUM_ENVS, STEPS = 3, 240

In [2]:
import warp as wp
wp.init()
import numpy as np, torch, yaml
from mjlab.tasks.registry import load_env_cfg
from mjlab.envs.manager_based_rl_env import ManagerBasedRlEnv

cfg = load_env_cfg(TASK)
cfg.scene.num_envs = NUM_ENVS
cfg.viewer.width, cfg.viewer.height = 960, 540
cfg.viewer.distance, cfg.viewer.elevation = 6.0, -20.0
env = ManagerBasedRlEnv(cfg, device='cuda', render_mode='rgb_array')
obs, _ = env.reset()
key = 'actor' if 'actor' in obs else 'policy'

Warp 1.14.0 initialized:
   CUDA Toolkit 12.9, Driver 13.2
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "NVIDIA GeForce RTX 4090" (24 GiB, sm_89, mempool enabled)
   Kernel cache:
     /home/viktor/.cache/warp/1.14.0


Module mujoco_warp._src.smooth 6acfd0b load on device 'cuda:0' took 3.55 ms  (cached)
Module _nxn_broadphase__locals__kernel_df6640ed df6640e load on device 'cuda:0' took 1.02 ms  (cached)


Module _primitive_narrowphase__locals__primitive_narrowphase_ad5f51d1 ad5f51d load on device 'cuda:0' took 1.43 ms  (cached)
Module mujoco_warp._src.constraint 880bd7f load on device 'cuda:0' took 0.34 ms  (cached)
Module _friction_dof__locals__kernel_3c14efda 3c14efd load on device 'cuda:0' took 1.74 ms  (cached)
Module _limit_slide_hinge__locals__kernel_4d6c55ab 4d6c55a load on device 'cuda:0' took 1.41 ms  (cached)
Module _efc_contact_init__locals__kernel_744a1fda 744a1fd load on device 'cuda:0' took 2.24 ms  (cached)
Module _efc_contact_jac_dense__locals__kernel_800e572c c417494 load on device 'cuda:0' took 0.62 ms  (cached)
Module _efc_contact_update__locals__kernel_c40b0799 c40b079 load on device 'cuda:0' took 0.54 ms  (cached)
Module mujoco_warp._src.sensor 4bf7c4d load on device 'cuda:0' took 2.94 ms  (cached)
Module mujoco_warp._src.forward a27d81a load on device 'cuda:0' took 1.13 ms  (cached)
Module mujoco_warp._src.passive e551f58 load on device 'cuda:0' took 1.12 ms  (cach

Module _tile_cholesky_factorize_solve_block__locals__kernel_419fe7e0 4a5ffaf load on device 'cuda:0' took 42.82 ms  (cached)
Module mujoco_warp._src.solver ee39d44 load on device 'cuda:0' took 3.12 ms  (cached)
Module _solve_init_jaref_kernel__locals__kernel_5a4f7ef7 5a4f7ef load on device 'cuda:0' took 0.47 ms  (cached)
Module mul_m_kernel__locals___mul_m_fd2186c4 fd2186c load on device 'cuda:0' took 0.47 ms  (cached)
Module _update_constraint_efc__locals__kernel_d40ce3fb d40ce3f load on device 'cuda:0' took 0.45 ms  (cached)
Module _update_gradient_JTDAJ_dense_tiled__locals__kernel_d367b9e1 476814b load on device 'cuda:0' took 1.01 ms  (cached)
Module _update_gradient_cholesky__locals__kernel_5c291ae5 6445ad4 load on device 'cuda:0' took 44.20 ms  (cached)
Module _linesearch_iterative_kernel__locals__kernel_3723f0ce 1b62d77 load on device 'cuda:0' took 1.03 ms  (cached)
Module _contact_sort__locals__contact_sort_321f5f3e 33ea4e1 load on device 'cuda:0' took 1.00 ms  (cached)
Module m

Module reset_data__locals__reset_xfrc_applied_03030511 0303051 load on device 'cuda:0' took 0.78 ms  (cached)
Module reset_data__locals__reset_M_631075dd 631075d load on device 'cuda:0' took 2.62 ms  (cached)
Module reset_data__locals__reset_mocap_6513201c 6513201 load on device 'cuda:0' took 0.64 ms  (cached)
Module reset_data__locals__reset_contact_5c08f689 5c08f68 load on device 'cuda:0' took 0.36 ms  (cached)
Module reset_data__locals__reset_sleep_bd8c7e06 bd8c7e0 load on device 'cuda:0' took 0.35 ms  (cached)
Module reset_data__locals__reset_nworld_092d1d28 092d1d2 load on device 'cuda:0' took 0.41 ms  (cached)


Module mujoco_warp._src.render_util ff4fe80 load on device 'cuda:0' took 0.62 ms  (cached)
Module mujoco_warp._src.bvh d9cfd91 load on device 'cuda:0' took 1.09 ms  (cached)
Module mujoco_warp._src.ray d79c433 load on device 'cuda:0' took 1.34 ms  (cached)

+--------------------------------+
|        Base Environment        |
+------------------------+-------+
| Property               | Value |
+------------------------+-------+
| Number of environments | 3     |
| Environment device     | cuda  |
| Environment seed       | None  |
| Physics step-size      | 0.005 |
| Environment step-size  | 0.02  |
+------------------------+-------+



[INFO] <EventManager> contains 3 active terms.
+-------------------------------------+
| Active Event Terms in Mode: 'reset' |
+--------+----------------------------+
| Index  | Name                       |
+--------+----------------------------+
|   0    | reset_base                 |
|   1    | reset_robot_joints         |
+--------+----------------------------+
+----------------------------------------------+
|    Active Event Terms in Mode: 'interval'    |
+-------+------------+-------------------------+
| Index | Name       | Interval time range (s) |
+-------+------------+-------------------------+
|   0   | push_robot |        (1.0, 3.0)       |
+-------+------------+-------------------------+
+---------------------------------------+
| Active Event Terms in Mode: 'startup' |
+---------+-----------------------------+
|  Index  | Name                        |
+---------+-----------------------------+
|    0    | encoder_bias                |
|    1    | base_com                  


Module mujoco_warp._src.io 5068124 load on device 'cuda:0' took 2.31 ms  (cached)
Module _tile_cholesky_factorize_block__locals__kernel_2efc24b8 2b5140f load on device 'cuda:0' took 29.77 ms  (cached)
Module _tile_cholesky_solve_block__locals__kernel_8ea2ce58 1256b29 load on device 'cuda:0' took 29.85 ms  (cached)


In [3]:
# Build an rl_games player from the training config + checkpoint
from gymnasium import spaces
from rl_games.torch_runner import Runner

with open(CONFIG) as f:
    params = yaml.safe_load(f)['params']
params['config']['env_info'] = {
    'observation_space': spaces.Box(-np.inf, np.inf, (obs[key].shape[-1],), np.float32),
    'action_space': spaces.Box(-1.0, 1.0, (env.action_space.shape[-1],), np.float32),
    'agents': 1,
}
runner = Runner(); runner.load({'params': params})
player = runner.create_player()
player.restore(CHECKPOINT)
player.has_batch_dimension = True

self.seed = 42
build mlp: 48
RunningMeanStd:  (1,)
RunningMeanStd:  (48,)
=> loading checkpoint 'runs/MJLab_Go1_notebook_17-22-17-29/nn/MJLab_Go1_notebook.pth'


In [4]:
# Rollout + inline video
import imageio
frames = []
for t in range(STEPS):
    with torch.no_grad():
        action = player.get_action(obs[key], is_deterministic=True)
    obs, rew, term, trunc, info = env.step(action)
    frames.append(env.render())
imageio.mimwrite('mjlab_rollout.mp4', frames, fps=30, quality=8)

from IPython.display import Video
Video('mjlab_rollout.mp4', embed=True, width=720)

IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (960, 540) to (960, 544) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


In [5]:
# Velocity probe: pin a forward command, measure achieved base velocity.
# Healthy Go1 at commanded 1.0 achieves ~0.85-1.0; a "stander" achieves ~0.
COMMANDED_VX = 1.0
cmd = env.unwrapped.command_manager.get_term('twist')
obs, _ = env.reset()
vels = []
for t in range(STEPS):
    cmd.vel_command_b[:, 0] = COMMANDED_VX
    cmd.vel_command_b[:, 1:] = 0.0
    cmd.is_heading_env[:] = False; cmd.is_standing_env[:] = False
    cmd.is_world_env[:] = False; cmd.is_forward_env[:] = False
    with torch.no_grad():
        action = player.get_action(obs[key], is_deterministic=True)
    obs, rew, term, trunc, info = env.step(action)
    if t >= STEPS // 2:
        vels.append(cmd.robot.data.root_link_lin_vel_b[:, 0].clone())
v = torch.stack(vels)
print(f"commanded vx={COMMANDED_VX:.2f} -> achieved {v.mean().item():.3f} "
      f"+/- {v.std().item():.3f} m/s")
env.close()

commanded vx=1.00 -> achieved 0.815 +/- 0.077 m/s


More elaborate demo scripting (multi-segment command sweeps, camera-aware headings,
goal visualization) and headless batch rendering: see `docs/MJLAB.md`.